# Energy Intelligence Platform

## Notebook 02 - RAG Chat Engine

Este notebook implementa el motor conversacional basado en Retrieval-Augmented Generation (RAG).

Componentes:

1. Carga del índice vectorial generado en Notebook 01.
2. Recuperación semántica mediante FAISS.
3. Generación de respuestas con Groq.
4. Citación de fuentes documentales.

Artefactos utilizados:

- energy_rag.index
- energy_chunks.pkl

Este notebook servirá posteriormente como base para:

- Notebook 03 (Sistema Multi-Agente)
- API FastAPI
- Integración con Lovable

## 1. Instalación

In [1]:
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.6 MB/s eta 0:00:00


## 2. Configuracion

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
BASE_PATH = "/content/drive/MyDrive/EnergyRag"

INDEX_PATH = f"{BASE_PATH}/energy_rag.index"
CHUNKS_PATH = f"{BASE_PATH}/energy_chunks.pkl"

print(INDEX_PATH)
print(CHUNKS_PATH)

/content/drive/MyDrive/EnergyRag/energy_rag.index
/content/drive/MyDrive/EnergyRag/energy_chunks.pkl


## 3. Carga de FAISS

In [4]:
import faiss

index = faiss.read_index(INDEX_PATH)

print("Vectores cargados:", index.ntotal)

Vectores cargados: 9986


## 4. Carga de Chunks

In [5]:
import pickle

with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)

print("Chunks cargados:", len(chunks))

Chunks cargados: 9986


## 5. Carga de modelo embeddings

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Modelo cargado")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado


## 6. Conectamos Groq

In [8]:
from groq import Groq
import os

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

print("Groq conectado")

Groq conectado


### 6.1 Funcion de busqueda semántica

In [9]:
import numpy as np

def search(query, k=5):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

### 6.2 Construccion del contexto

In [10]:
def build_context(results):

    context = "\n\n".join(
        [
            f"[SOURCE: {r['source']}]\n{r['text']}"
            for r in results
        ]
    )

    return context

### 6.3 Funcion Rag

In [11]:
def answer_question(question):

    retrieved = search(
        question,
        k=5
    )

    context = build_context(
        retrieved
    )

    prompt = f"""
You are an Energy Intelligence Assistant.

Answer ONLY using the information contained in the context.

If the answer is not present, say:
"I cannot find this information in the knowledge base."

CONTEXT:

{context}

QUESTION:

{question}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    return {
        "answer": answer,
        "sources": list(
            set(
                [r["source"] for r in retrieved]
            )
        )
    }

In [12]:
# primera prueba
result = answer_question(
    "What is the role of energy storage in the energy transition?"
)

print(result["answer"])

print("\nSOURCES:\n")

for s in result["sources"]:
    print("-", s)

Energy storage, such as batteries, plays a crucial role in the energy transition by allowing for the absorption of wind power and other intermittent renewable energy sources, improving the investment case for renewables, and enabling the delivery of reliable and affordable electricity. It helps to balance generation and consumption, reduce curtailment, and smooth peaks, thereby maintaining reliability in the power system.

SOURCES:

- 1_World_Energy_Outlook_2025.pdf
- 4_Global_Wind_Report_2026.pdf
- 12_EnergyandAI.pdf


In [13]:
# prueba España
result = answer_question(
    "What are the main objectives of Spain's PNIEC 2030?"
)

print(result["answer"])

print("\nSOURCES:\n")

for s in result["sources"]:
    print("-", s)

I cannot find this information in the knowledge base.

SOURCES:

- 13_PNIEC_España_2024_240924.pdf


In [14]:
result = answer_question(
    "How is artificial intelligence impacting electricity demand?"
)

print(result["answer"])

print("\nSOURCES:\n")

for s in result["sources"]:
    print("-", s)

Artificial intelligence (AI) is having a significant impact on electricity demand. Rapidly growing investment in data centres is already straining grids in some places and raising concerns about the ability of the electricity system to meet a surge in demand. However, the exact share of data centre electricity demand that comes from AI is challenging to answer due to the lack of comprehensive data on the share of different kinds of workloads and limited visibility over specific workloads running in data centre facilities. Additionally, electricity demand is already growing strongly in emerging market and developing economies, driven by economic growth, industrialisation, and increased adoption of appliances, which may be further exacerbated by AI adoption.

SOURCES:

- 12_EnergyandAI.pdf
